# 🌍 Week 3 — JSON and File Processing
### Geospatial Python Mastery | Module B: Data Structures and File I/O

---

|  |  |
|---|---|
| **Course** | Geospatial Python Mastery |
| **Week** | 3 of 10 |
| **Theme** | JSON and File Processing for Geospatial Data |
| **Duration** | ~4 contact hours + 4 hours self-study |
| **Practice Outcome** | Mini-Lab: Build a robust GeoJSON validator and batch processor |
| **Next week** | Geospatial foundations — coordinates, projections, CRS theory |

---

> 🗺️ **Why this week matters**
> Geospatial analysis depends on reliable file handling. GeoJSON, CSV, logs, and test files
> are the plumbing behind every later workflow in GeoPandas, PostGIS, and CityJSON.
> This week turns raw text files into structured Python objects, validates coordinates and
> geometry types, and packages the logic into reusable functions you can trust.

---

## 📋 Table of Contents

| Section | Topic |
|---------|-------|
| [1 — Environment Setup](#section-1) | Auto-install, imports, data dirs |
| [2 — pathlib Basics](#section-2) | Path objects, mkdir, glob, read_text/write_text |
| [3 — Working with CSV Files](#section-3) | DictReader, DictWriter, coordinate validation |
| [4 — JSON and GeoJSON](#section-4) | loads, dumps, FeatureCollection structure |
| [5 — GeoJSON Validation](#section-5) | Coordinate checks and geometry rules |
| [6 — Building a Reusable Module](#section-6) | Write and import `geo_file_tools.py` |
| [7 — Error Handling](#section-7) | Custom exceptions and resilient pipelines |
| [8 — Logging](#section-8) | FileHandler, levels, audit trail |
| [9 — Unit Testing](#section-9) | `unittest.TestCase`, setUp, assertRaises |
| [10 — Batch Processing](#section-10) | Process many files and save a summary |

---

## 🗺️ Symbol Guide

| Symbol | Meaning |
|--------|---------|
| 💻 | Runnable code cell |
| 🎯 | Exercise — write your own code |
| ✅ | Solution — run after attempting |
| 🔬 | Mini-Lab step |
| 📖 | Explanatory section |
| 💡 | Tip or best practice |
| ⚠️ | Common mistake or warning |

> **Keyboard shortcuts:** `Shift+Enter` run cell · `b` insert cell below · `m` convert to Markdown · `Esc` command mode

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

| # | Objective |
|---|-----------|
| 1 | Use Python's `pathlib` module to manage file paths portably across operating systems |
| 2 | Read and write **CSV** files with the `csv` module (`DictReader`, `DictWriter`) |
| 3 | Parse and generate **JSON** and **GeoJSON** programmatically with the `json` module |
| 4 | Validate coordinate ranges and geometry types in GeoJSON `FeatureCollection` objects |
| 5 | Build a reusable `geo_file_tools.py` module with `load_features`, `extract_valid_points`, and `save_geojson` |
| 6 | Handle file I/O errors with `try/except` and custom exceptions |
| 7 | Add structured **logging** to a file-processing pipeline |
| 8 | Write `unittest` tests for file utility functions |

### How to use this notebook

- Run cells **top to bottom** in sequence
- **Attempt** each 🎯 exercise before looking at the ✅ solution
- The 🔬 **Mini-Lab** at the end ties all skills together
- Tick each box in the ☑️ checklist before moving to the next week

In [2]:
# 💻 Environment check and auto-install
# ─────────────────────────────────────────────────────────────────────────────
import sys, subprocess, importlib, platform

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

OS_NAME = platform.system()

print("=" * 60)
print("  Geospatial Python Mastery — Week 3 Environment Check")
print("=" * 60)
print(f"  Environment : {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"  Python      : {sys.version.split()[0]}")
print(f"  OS          : {OS_NAME}")

REQUIRED = [
    ("pathlib",  "",          ""),
    ("csv",      "",          ""),
    ("json",     "",          ""),
    ("logging",  "",          ""),
    ("unittest", "",          ""),
    ("uuid",     "",          ""),
]

for import_name, pip_name, min_ver in REQUIRED:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, "__version__", "stdlib")
        print(f"  ✅ {import_name:<20} {ver}")
    except ImportError:
        pkg = pip_name or import_name
        print(f"  ⬇  Installing {pkg}…")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"  ✅ {pkg} installed")

if OS_NAME == "Windows":
    print("\n  ℹ  Windows note: use 'python -m jupyter notebook' to launch Jupyter")

print("\n✅ Environment check complete — ready for Week 3!")

  Geospatial Python Mastery — Week 3 Environment Check
  Environment : Google Colab
  Python      : 3.12.13
  OS          : Linux
  ✅ pathlib              stdlib
  ✅ csv                  1.0
  ✅ json                 2.0.9
  ✅ logging              0.5.1.2
  ✅ unittest             stdlib
  ✅ uuid                 stdlib

✅ Environment check complete — ready for Week 3!


In [3]:
# 💻 0.1  Import core libraries used this week
# ─────────────────────────────────────────────────────────────────────────────
import csv
import json
import logging
import sys
import unittest
import uuid
from importlib import reload
from pathlib import Path
from pprint import pprint

print('Imports loaded successfully.')

Imports loaded successfully.


In [4]:
# 💻 0.2  Create data, logs, modules, and test folders
# ─────────────────────────────────────────────────────────────────────────────
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'notebooks').exists() and (candidate / 'instructions').exists():
            return candidate
    return start

PROJECT_ROOT = find_repo_root(Path.cwd())
COURSE_DIR = PROJECT_ROOT / 'notebooks' / 'geospatial_python_course'
WEEK_DIR = COURSE_DIR / 'data' / 'week_03'
INPUT_DIR = WEEK_DIR / 'input'
OUTPUT_DIR = WEEK_DIR / 'output'
LOG_DIR = COURSE_DIR / 'logs'
MODULE_DIR = COURSE_DIR / 'local_modules'
TEST_DIR = COURSE_DIR / 'tests'

for path in [WEEK_DIR, INPUT_DIR, OUTPUT_DIR, LOG_DIR, MODULE_DIR, TEST_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Course dir   :', COURSE_DIR)
print('Week data    :', WEEK_DIR)
print('Output dir   :', OUTPUT_DIR)
print('Log file dir :', LOG_DIR)

Project root : /content
Course dir   : /content/notebooks/geospatial_python_course
Week data    : /content/notebooks/geospatial_python_course/data/week_03
Output dir   : /content/notebooks/geospatial_python_course/data/week_03/output
Log file dir : /content/notebooks/geospatial_python_course/logs


---
## 📖 Section 1 — Environment Setup

Start with a stable workspace: known folders, predictable filenames, and reproducible sample data.

### 1.1 Project folders and starter files
Everything this week lives under `data/week_03/`, while reusable modules and unit tests stay beside the notebook for easy imports.

In [5]:
# 💻 1.1  Inspect the workspace layout
# ─────────────────────────────────────────────────────────────────────────────
workspace = {
    'input': INPUT_DIR,
    'output': OUTPUT_DIR,
    'logs': LOG_DIR,
    'modules': MODULE_DIR,
    'tests': TEST_DIR,
}

for label, path in workspace.items():
    print(f'{label:<8} -> {path}')

print('\nCurrent contents of the input directory:')
print(list(INPUT_DIR.iterdir()))

input    -> /content/notebooks/geospatial_python_course/data/week_03/input
output   -> /content/notebooks/geospatial_python_course/data/week_03/output
logs     -> /content/notebooks/geospatial_python_course/logs
modules  -> /content/notebooks/geospatial_python_course/local_modules
tests    -> /content/notebooks/geospatial_python_course/tests

Current contents of the input directory:
[]


In [6]:
# 💻 1.2  Seed example CSV and GeoJSON files
# ─────────────────────────────────────────────────────────────────────────────
seed_rows = [
    {'city': 'Amsterdam', 'lon': '4.9041', 'lat': '52.3676', 'country': 'NL'},
    {'city': 'Rotterdam', 'lon': '4.4777', 'lat': '51.9244', 'country': 'NL'},
    {'city': 'BadTown', 'lon': '190.0000', 'lat': '95.0000', 'country': '??'},
]

sample_csv_path = INPUT_DIR / 'city_points.csv'
with sample_csv_path.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['city', 'lon', 'lat', 'country'])
    writer.writeheader()
    writer.writerows(seed_rows)

starter_geojson = {
    'type': 'FeatureCollection',
    'features': [
        {
            'type': 'Feature',
            'properties': {'city': 'Amsterdam', 'country': 'NL'},
            'geometry': {'type': 'Point', 'coordinates': [4.9041, 52.3676]},
        },
        {
            'type': 'Feature',
            'properties': {'city': 'Rotterdam', 'country': 'NL'},
            'geometry': {'type': 'Point', 'coordinates': [4.4777, 51.9244]},
        },
        {
            'type': 'Feature',
            'properties': {'city': 'BadTown', 'country': '??'},
            'geometry': {'type': 'Point', 'coordinates': [190.0, 95.0]},
        },
    ],
}

starter_geojson_path = INPUT_DIR / 'starter_points.geojson'
starter_geojson_path.write_text(json.dumps(starter_geojson, indent=2), encoding='utf-8')
run_manifest = {
    'week': 3,
    'run_id': uuid.uuid4().hex[:8],
    'input_files': [sample_csv_path.name, starter_geojson_path.name],
}
(INPUT_DIR / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')

print('Wrote:', sample_csv_path.name, starter_geojson_path.name, 'and run_manifest.json')

Wrote: city_points.csv starter_points.geojson and run_manifest.json


### 🎯 Exercise 1 — Create a scratch workspace

**Task:** Use `pathlib` to create a scratch folder inside `INPUT_DIR`, then write a JSON manifest describing the files you want to process.

**Steps:**
1. Create `INPUT_DIR / "scratch"` if it does not already exist.
2. Write a `scratch_manifest.json` file with a short description and a list of filenames.

```python
# Hint
scratch_dir = INPUT_DIR / 'scratch'
```

In [ ]:
# 🎯 Exercise 1 — your code here ────────────────────────────────────────────
# 1. Create the directory.
# 2. Write the manifest JSON file.

In [ ]:
# ✅ Exercise 1 — Solution ────────────────────────────────────────────────────
scratch_dir = INPUT_DIR / 'scratch'
scratch_dir.mkdir(parents=True, exist_ok=True)

scratch_manifest = {
    'description': 'Temporary staging area for Week 3 file practice.',
    'files': ['city_points.csv', 'starter_points.geojson'],
}
manifest_path = scratch_dir / 'scratch_manifest.json'
manifest_path.write_text(json.dumps(scratch_manifest, indent=2), encoding='utf-8')

print('Scratch manifest saved to:', manifest_path)
print(manifest_path.read_text(encoding='utf-8'))

---
## 📖 Section 2 — pathlib Basics

`Path` objects replace fragile string concatenation and make your scripts portable.

### 2.1 Joining paths, checking files, and discovering data
With `pathlib` you can create directories, iterate over files, and read or write text without worrying about slash direction.

In [7]:
# 💻 2.1  Inspect Path object behaviour
# ─────────────────────────────────────────────────────────────────────────────
print('Sample CSV name   :', sample_csv_path.name)
print('Sample CSV suffix :', sample_csv_path.suffix)
print('Sample CSV parent :', sample_csv_path.parent)
print('Resolved path     :', sample_csv_path.resolve())

archive_path = OUTPUT_DIR / 'archive' / 'city_points_backup.csv'
archive_path.parent.mkdir(parents=True, exist_ok=True)
print('Joined path       :', archive_path)

Sample CSV name   : city_points.csv
Sample CSV suffix : .csv
Sample CSV parent : /content/notebooks/geospatial_python_course/data/week_03/input
Resolved path     : /content/notebooks/geospatial_python_course/data/week_03/input/city_points.csv
Joined path       : /content/notebooks/geospatial_python_course/data/week_03/output/archive/city_points_backup.csv


In [8]:
# 💻 2.2  Use glob, write_text, and read_text
# ─────────────────────────────────────────────────────────────────────────────
notes_path = OUTPUT_DIR / 'week3_notes.txt'
notes_path.write_text(
    'Week 3 focuses on pathlib, CSV, JSON, GeoJSON, logging, and unittest.\n',
    encoding='utf-8',
)

print('Files in input folder:')
for path in sorted(INPUT_DIR.glob('*')):
    print(' -', path.name)

print('\nSaved note contents:')
print(notes_path.read_text(encoding='utf-8'))

Files in input folder:
 - city_points.csv
 - run_manifest.json
 - starter_points.geojson

Saved note contents:
Week 3 focuses on pathlib, CSV, JSON, GeoJSON, logging, and unittest.



### 🎯 Exercise 2 — Preview every GeoJSON file

**Task:** Discover all `.geojson` files in `INPUT_DIR` and print the first 120 characters from each file.

**Steps:**
1. Use `INPUT_DIR.glob("*.geojson")` to get candidate files.
2. Loop over the files and print a short preview from `read_text()`.

```python
# Hint
for path in INPUT_DIR.glob("*.geojson"):
    ...
```

In [ ]:
# 🎯 Exercise 2 — your code here ────────────────────────────────────────────
# 1. Find all GeoJSON files.
# 2. Read and preview each file.

In [ ]:
# ✅ Exercise 2 — Solution ────────────────────────────────────────────────────
for path in sorted(INPUT_DIR.glob('*.geojson')):
    preview = path.read_text(encoding='utf-8')[:120]
    print(f'\n{path.name}')
    print(preview + '...')

---
## 📖 Section 3 — Working with CSV Files

CSV files are common for tabular location data because they are simple, transparent, and easy to exchange.

### 3.1 Reading rows into dictionaries
Use `csv.DictReader` when your file has headers. It returns each row as a dict, which makes later validation and conversion much easier.

In [9]:
# 💻 3.1  Read CSV rows and coerce lon/lat values
# ─────────────────────────────────────────────────────────────────────────────
def coordinate_is_valid(lon: float, lat: float) -> bool:
    return -180 <= lon <= 180 and -90 <= lat <= 90

csv_records = []
with sample_csv_path.open('r', newline='', encoding='utf-8') as fh:
    reader = csv.DictReader(fh)
    for row in reader:
        lon = float(row['lon'])
        lat = float(row['lat'])
        record = {
            'city': row['city'],
            'country': row['country'],
            'lon': lon,
            'lat': lat,
            'is_valid': coordinate_is_valid(lon, lat),
        }
        csv_records.append(record)

pprint(csv_records)

[{'city': 'Amsterdam',
  'country': 'NL',
  'is_valid': True,
  'lat': 52.3676,
  'lon': 4.9041},
 {'city': 'Rotterdam',
  'country': 'NL',
  'is_valid': True,
  'lat': 51.9244,
  'lon': 4.4777},
 {'city': 'BadTown',
  'country': '??',
  'is_valid': False,
  'lat': 95.0,
  'lon': 190.0}]


In [10]:
# 💻 3.2  Write cleaned and rejected CSV files
# ─────────────────────────────────────────────────────────────────────────────
clean_csv_path = OUTPUT_DIR / 'city_points_clean.csv'
rejected_csv_path = OUTPUT_DIR / 'city_points_rejected.csv'
fieldnames = ['city', 'country', 'lon', 'lat', 'is_valid']

with clean_csv_path.open('w', newline='', encoding='utf-8') as clean_fh, rejected_csv_path.open('w', newline='', encoding='utf-8') as bad_fh:
    clean_writer = csv.DictWriter(clean_fh, fieldnames=fieldnames)
    bad_writer = csv.DictWriter(bad_fh, fieldnames=fieldnames)
    clean_writer.writeheader()
    bad_writer.writeheader()
    for record in csv_records:
        target = clean_writer if record['is_valid'] else bad_writer
        target.writerow(record)

print('Clean CSV   :', clean_csv_path)
print('Rejected CSV:', rejected_csv_path)

Clean CSV   : /content/notebooks/geospatial_python_course/data/week_03/output/city_points_clean.csv
Rejected CSV: /content/notebooks/geospatial_python_course/data/week_03/output/city_points_rejected.csv


### 🎯 Exercise 3 — Build a validated CSV export

**Task:** Read `city_points.csv`, convert longitude and latitude strings to floats, and write a new CSV containing only valid rows.

**Steps:**
1. Use `csv.DictReader` to iterate over rows.
2. Convert `lon` and `lat` to floats and keep only valid coordinate pairs.

```python
# Hint
valid_rows = []
with sample_csv_path.open(...) as fh:
    ...
```

In [ ]:
# 🎯 Exercise 3 — your code here ────────────────────────────────────────────
# 1. Read rows from the CSV file.
# 2. Write only valid rows to a new CSV file.

In [ ]:
# ✅ Exercise 3 — Solution ────────────────────────────────────────────────────
valid_rows = []
with sample_csv_path.open('r', newline='', encoding='utf-8') as fh:
    reader = csv.DictReader(fh)
    for row in reader:
        lon = float(row['lon'])
        lat = float(row['lat'])
        if coordinate_is_valid(lon, lat):
            valid_rows.append({'city': row['city'], 'country': row['country'], 'lon': lon, 'lat': lat})

validated_csv_path = OUTPUT_DIR / 'validated_points.csv'
with validated_csv_path.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['city', 'country', 'lon', 'lat'])
    writer.writeheader()
    writer.writerows(valid_rows)

print('Validated rows:', len(valid_rows))
print('Saved to       :', validated_csv_path)

---
## 📖 Section 4 — JSON and GeoJSON

JSON gives structure to nested data, while GeoJSON adds a standard way to describe geometry and properties.

### 4.1 Parsing JSON text and inspecting a FeatureCollection
A GeoJSON FeatureCollection is just JSON with agreed keys like `type`, `features`, `properties`, and `geometry`.

In [ ]:
# 💻 4.1  Load JSON text into Python dictionaries
# ─────────────────────────────────────────────────────────────────────────────
raw_geojson_text = starter_geojson_path.read_text(encoding='utf-8')
parsed_geojson = json.loads(raw_geojson_text)

print('Top-level type :', parsed_geojson['type'])
print('Feature count  :', len(parsed_geojson['features']))
print('\nFirst feature pretty-print:')
print(json.dumps(parsed_geojson['features'][0], indent=2))

In [ ]:
# 💻 4.2  Generate GeoJSON programmatically from CSV rows
# ─────────────────────────────────────────────────────────────────────────────
geojson_from_csv = {
    'type': 'FeatureCollection',
    'features': [
        {
            'type': 'Feature',
            'properties': {'city': row['city'], 'country': row['country']},
            'geometry': {'type': 'Point', 'coordinates': [row['lon'], row['lat']]},
        }
        for row in csv_records
        if row['is_valid']
    ],
}

clean_geojson_path = OUTPUT_DIR / 'clean_points.geojson'
clean_geojson_path.write_text(json.dumps(geojson_from_csv, indent=2), encoding='utf-8')
print('Saved clean GeoJSON to:', clean_geojson_path)

### 🎯 Exercise 4 — Create a one-feature GeoJSON object

**Task:** Turn a single city dictionary into a valid GeoJSON `FeatureCollection` and pretty-print it.

**Steps:**
1. Start from a small Python dict containing `city`, `country`, `lon`, and `lat`.
2. Wrap the resulting feature inside a `FeatureCollection` with one entry.

```python
# Hint
feature = {"type": "Feature", ...}
```

In [ ]:
# 🎯 Exercise 4 — your code here ────────────────────────────────────────────
# 1. Create one feature dict.
# 2. Wrap it in a FeatureCollection and print it.

In [ ]:
# ✅ Exercise 4 — Solution ────────────────────────────────────────────────────
city_record = {'city': 'Utrecht', 'country': 'NL', 'lon': 5.1214, 'lat': 52.0907}
feature = {
    'type': 'Feature',
    'properties': {'city': city_record['city'], 'country': city_record['country']},
    'geometry': {'type': 'Point', 'coordinates': [city_record['lon'], city_record['lat']]},
}
one_feature_collection = {'type': 'FeatureCollection', 'features': [feature]}

print(json.dumps(one_feature_collection, indent=2))

---
## 📖 Section 5 — GeoJSON Validation

Validation protects downstream analysis from subtle geometry errors and impossible coordinate values.

### 5.1 Checking geometry type and coordinate ranges
For this week we focus on GeoJSON `Point` features, require a `Feature` wrapper, and enforce global WGS84 bounds.

In [ ]:
# 💻 5.1  Create a reusable feature validator
# ─────────────────────────────────────────────────────────────────────────────
class GeoJSONValidationError(Exception):
    """Raised when a GeoJSON feature breaks an expected rule."""

ALLOWED_GEOMETRY_TYPES = {'Point'}


def validate_feature(feature: dict) -> None:
    if feature.get('type') != 'Feature':
        raise GeoJSONValidationError('Feature must declare type="Feature".')

    geometry = feature.get('geometry') or {}
    geometry_type = geometry.get('type')
    if geometry_type not in ALLOWED_GEOMETRY_TYPES:
        raise GeoJSONValidationError(f'Unsupported geometry type: {geometry_type}')

    coordinates = geometry.get('coordinates')
    if not isinstance(coordinates, (list, tuple)) or len(coordinates) != 2:
        raise GeoJSONValidationError('Point coordinates must be a two-item list.')

    lon, lat = coordinates
    if not coordinate_is_valid(float(lon), float(lat)):
        raise GeoJSONValidationError(f'Coordinate out of range: {(lon, lat)}')

In [ ]:
# 💻 5.2  Validate a whole FeatureCollection
# ─────────────────────────────────────────────────────────────────────────────
def split_valid_and_invalid_features(collection: dict):
    valid_features = []
    invalid_features = []
    for feature in collection.get('features', []):
        try:
            validate_feature(feature)
            valid_features.append(feature)
        except GeoJSONValidationError as exc:
            invalid_features.append({'feature': feature, 'error': str(exc)})
    return valid_features, invalid_features

valid_features, invalid_details = split_valid_and_invalid_features(parsed_geojson)
print('Valid features  :', len(valid_features))
print('Invalid features:', len(invalid_details))
pprint(invalid_details)

### 🎯 Exercise 5 — Reject bad GeoJSON features

**Task:** Run `validate_feature()` against one good feature and one bad feature, then confirm that the bad one raises your custom exception.

**Steps:**
1. Create a valid point feature and an invalid point feature.
2. Use `try/except` to show the error message for the invalid one.

```python
# Hint
try:
    validate_feature(bad_feature)
except GeoJSONValidationError as exc:
    ...
```

In [ ]:
# 🎯 Exercise 5 — your code here ────────────────────────────────────────────
# 1. Validate a good feature.
# 2. Catch the exception from a bad feature.

In [ ]:
# ✅ Exercise 5 — Solution ────────────────────────────────────────────────────
good_feature = {
    'type': 'Feature',
    'properties': {'city': 'Leiden'},
    'geometry': {'type': 'Point', 'coordinates': [4.4970, 52.1601]},
}
bad_feature = {
    'type': 'Feature',
    'properties': {'city': 'Broken'},
    'geometry': {'type': 'Point', 'coordinates': [222.0, 91.0]},
}

validate_feature(good_feature)
print('Good feature passed validation.')

try:
    validate_feature(bad_feature)
except GeoJSONValidationError as exc:
    print('Bad feature failed as expected:', exc)

---
## 📖 Section 6 — Building a Reusable Module

Notebooks are great for learning, but reusable modules keep logic tidy, testable, and importable.

### 6.1 Writing `geo_file_tools.py` from notebook code
We will export a small utility module that hides file parsing details behind clean function names.

In [ ]:
# 💻 6.1  Write the geo_file_tools.py module
# ─────────────────────────────────────────────────────────────────────────────
geo_file_tools_path = MODULE_DIR / 'geo_file_tools.py'
geo_file_tools_source = """
import json
from pathlib import Path


class GeoFileToolsError(Exception):
    pass


class GeoJSONValidationError(GeoFileToolsError):
    pass


ALLOWED_GEOMETRY_TYPES = {'Point'}


def _validate_feature(feature: dict) -> None:
    if feature.get('type') != 'Feature':
        raise GeoJSONValidationError('Expected a GeoJSON Feature object.')
    geometry = feature.get('geometry') or {}
    if geometry.get('type') not in ALLOWED_GEOMETRY_TYPES:
        raise GeoJSONValidationError('Only Point geometry is supported in Week 3.')
    coords = geometry.get('coordinates')
    if not isinstance(coords, (list, tuple)) or len(coords) != 2:
        raise GeoJSONValidationError('Point coordinates must contain lon and lat.')
    lon, lat = coords
    if not (-180 <= float(lon) <= 180 and -90 <= float(lat) <= 90):
        raise GeoJSONValidationError(f'Coordinates out of range: {(lon, lat)}')


def load_features(path):
    path = Path(path)
    payload = json.loads(path.read_text(encoding='utf-8'))
    return payload.get('features', [])


def extract_valid_points(features):
    valid = []
    for feature in features:
        _validate_feature(feature)
        valid.append(feature)
    return valid


def save_geojson(features, path):
    path = Path(path)
    collection = {'type': 'FeatureCollection', 'features': list(features)}
    path.write_text(json.dumps(collection, indent=2), encoding='utf-8')
    return path
"""
geo_file_tools_path.write_text(geo_file_tools_source.strip() + '\n', encoding='utf-8')
print('Module written to:', geo_file_tools_path)

In [ ]:
# 💻 6.2  Import and use the module functions
# ─────────────────────────────────────────────────────────────────────────────
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import geo_file_tools
geo_file_tools = reload(geo_file_tools)

module_loaded_features = geo_file_tools.load_features(clean_geojson_path)
module_valid_points = geo_file_tools.extract_valid_points(module_loaded_features)
module_output_path = OUTPUT_DIR / 'module_valid_points.geojson'
geo_file_tools.save_geojson(module_valid_points, module_output_path)

print('Features loaded :', len(module_loaded_features))
print('Valid points    :', len(module_valid_points))
print('Saved output to :', module_output_path)

### 🎯 Exercise 6 — Reuse your local module

**Task:** Import `geo_file_tools.py`, load features from `clean_points.geojson`, and save only the first two features to a new file.

**Steps:**
1. Import the module from `MODULE_DIR`.
2. Slice the loaded features list and pass it to `save_geojson()`.

```python
# Hint
subset = features[:2]
geo_file_tools.save_geojson(subset, OUTPUT_DIR / "subset.geojson")
```

In [ ]:
# 🎯 Exercise 6 — your code here ────────────────────────────────────────────
# 1. Import the module and load the features.
# 2. Save a subset to a new GeoJSON file.

In [ ]:
# ✅ Exercise 6 — Solution ────────────────────────────────────────────────────
subset_features = module_loaded_features[:2]
subset_output_path = OUTPUT_DIR / 'subset_points.geojson'
geo_file_tools.save_geojson(subset_features, subset_output_path)

print('Subset feature count:', len(subset_features))
print('Subset path         :', subset_output_path)

---
## 📖 Section 7 — Error Handling

Broken paths and malformed JSON are normal in real data pipelines, so your code should fail clearly instead of mysteriously.

### 7.1 Custom exceptions for missing files and bad JSON
A small exception hierarchy gives you better control over how to stop, retry, or log a failure.

In [ ]:
# 💻 7.1  Wrap file problems in named exceptions
# ─────────────────────────────────────────────────────────────────────────────
class GeoFilePipelineError(Exception):
    pass


class MissingInputFileError(GeoFilePipelineError):
    pass


class InvalidJSONError(GeoFilePipelineError):
    pass


def read_json_file(path: Path) -> dict:
    if not path.exists():
        raise MissingInputFileError(f'Missing input file: {path.name}')
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError as exc:
        raise InvalidJSONError(f'Bad JSON in {path.name}: {exc.msg}') from exc

In [ ]:
# 💻 7.2  Handle failures without stopping the whole pipeline
# ─────────────────────────────────────────────────────────────────────────────
broken_json_path = INPUT_DIR / 'broken.geojson'
broken_json_path.write_text('{"type": "FeatureCollection", bad json}', encoding='utf-8')

for candidate in [starter_geojson_path, broken_json_path, INPUT_DIR / 'missing.geojson']:
    try:
        payload = read_json_file(candidate)
        print(f'Loaded {candidate.name}: {payload.get("type", "unknown")}')
    except GeoFilePipelineError as exc:
        print(f'Handled error for {candidate.name}: {exc}')

### 🎯 Exercise 7 — Catch JSON failures gracefully

**Task:** Attempt to read a missing file and a malformed JSON file, then print clear messages instead of crashing.

**Steps:**
1. Call `read_json_file()` inside a `try/except` block.
2. Catch `GeoFilePipelineError` and print the exception text.

```python
# Hint
try:
    read_json_file(path)
except GeoFilePipelineError as exc:
    print(exc)
```

In [ ]:
# 🎯 Exercise 7 — your code here ────────────────────────────────────────────
# 1. Try to read a file.
# 2. Catch and print the error.

In [ ]:
# ✅ Exercise 7 — Solution ────────────────────────────────────────────────────
for path in [INPUT_DIR / 'missing_again.geojson', broken_json_path]:
    try:
        read_json_file(path)
    except GeoFilePipelineError as exc:
        print('Graceful failure:', exc)

---
## 📖 Section 8 — Logging

Logs create a timeline of what happened, which file was processed, and which records failed validation.

### 8.1 Configuring console and file logging
Use a `FileHandler` to persist diagnostics and a formatter to make each line easy to scan later.

In [ ]:
# 💻 8.1  Configure a logger for Week 3
# ─────────────────────────────────────────────────────────────────────────────
log_path = LOG_DIR / 'week3.log'
logger = logging.getLogger('gpm.week3')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

formatter = logging.Formatter('%(levelname)s | %(asctime)s | %(message)s')
stream_handler = logging.StreamHandler()
stream_handler.setLevel(logging.INFO)
stream_handler.setFormatter(formatter)

file_handler = logging.FileHandler(log_path, encoding='utf-8')
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

logger.addHandler(stream_handler)
logger.addHandler(file_handler)
logger.info('Week 3 logger initialised.')
print('Log file:', log_path)

In [ ]:
# 💻 8.2  Log a validation pass over the starter GeoJSON
# ─────────────────────────────────────────────────────────────────────────────
def log_validation_run(features, logger):
    summary = {'valid': 0, 'invalid': 0}
    for feature in features:
        city_name = feature.get('properties', {}).get('city', 'unknown')
        try:
            validate_feature(feature)
            summary['valid'] += 1
            logger.debug('Accepted feature for %s', city_name)
        except GeoJSONValidationError as exc:
            summary['invalid'] += 1
            logger.warning('Rejected feature for %s: %s', city_name, exc)
    logger.info('Validation summary: %s', summary)
    return summary

logged_summary = log_validation_run(parsed_geojson['features'], logger)
print(logged_summary)

### 🎯 Exercise 8 — Add warnings for invalid features

**Task:** Loop through the starter features and write a warning to `logs/week3.log` whenever a feature is invalid.

**Steps:**
1. Use `validate_feature()` inside a `try/except` block.
2. Call `logger.warning()` with the city name and error message.

```python
# Hint
logger.warning("Rejected %s: %s", city_name, exc)
```

In [ ]:
# 🎯 Exercise 8 — your code here ────────────────────────────────────────────
# 1. Loop over the features.
# 2. Log a warning when validation fails.

In [ ]:
# ✅ Exercise 8 — Solution ────────────────────────────────────────────────────
for feature in parsed_geojson['features']:
    city_name = feature.get('properties', {}).get('city', 'unknown')
    try:
        validate_feature(feature)
    except GeoJSONValidationError as exc:
        logger.warning('Exercise warning for %s: %s', city_name, exc)

print('Recent log lines:')
print('\n'.join(log_path.read_text(encoding='utf-8').splitlines()[-4:]))

---
## 📖 Section 9 — Unit Testing

Automated tests give you confidence that utility functions behave the same tomorrow as they do today.

### 9.1 Writing a `unittest.TestCase` for file helpers
We test happy paths and failure paths: successful loads, valid point extraction, and rejection of bad coordinates.

In [ ]:
# 💻 9.1  Write a test module for geo_file_tools.py
# ─────────────────────────────────────────────────────────────────────────────
test_file_path = TEST_DIR / 'test_geo_file_tools.py'
test_file_source = """
import sys
import unittest
from pathlib import Path

COURSE_DIR = Path(__file__).resolve().parents[1]
MODULE_DIR = COURSE_DIR / 'local_modules'
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import geo_file_tools


class TestGeoFileTools(unittest.TestCase):
    def setUp(self):
        self.good_feature = {
            'type': 'Feature',
            'properties': {'city': 'Delft'},
            'geometry': {'type': 'Point', 'coordinates': [4.3571, 52.0116]},
        }
        self.bad_feature = {
            'type': 'Feature',
            'properties': {'city': 'Broken'},
            'geometry': {'type': 'Point', 'coordinates': [999.0, 999.0]},
        }

    def test_extract_valid_points_returns_input_feature(self):
        result = geo_file_tools.extract_valid_points([self.good_feature])
        self.assertEqual(len(result), 1)
        self.assertEqual(result[0]['properties']['city'], 'Delft')

    def test_extract_valid_points_raises_on_bad_coords(self):
        with self.assertRaises(geo_file_tools.GeoJSONValidationError):
            geo_file_tools.extract_valid_points([self.bad_feature])

    def test_save_geojson_writes_feature_collection(self):
        out_dir = COURSE_DIR / 'tests' / '_artifacts'
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / 'test_points.geojson'
        geo_file_tools.save_geojson([self.good_feature], out_path)
        self.assertTrue(out_path.exists())
        self.assertIn('FeatureCollection', out_path.read_text(encoding='utf-8'))


if __name__ == '__main__':
    unittest.main()
"""
test_file_path.write_text(test_file_source.strip() + '\n', encoding='utf-8')
print('Wrote test file:', test_file_path)

In [ ]:
# 💻 9.2  Run the unittest suite from the notebook
# ─────────────────────────────────────────────────────────────────────────────
suite = unittest.defaultTestLoader.discover(str(TEST_DIR), pattern='test_geo_file_tools.py')
runner = unittest.TextTestRunner(verbosity=2)
test_result = runner.run(suite)
print('Successful:', test_result.wasSuccessful())

### 🎯 Exercise 9 — Write an assertRaises test

**Task:** Create a tiny `unittest.TestCase` that confirms `validate_feature()` raises `GeoJSONValidationError` for invalid coordinates.

**Steps:**
1. Subclass `unittest.TestCase`.
2. Write one method that uses `self.assertRaises(...)`.

```python
# Hint
with self.assertRaises(GeoJSONValidationError):
    validate_feature(bad_feature)
```

In [ ]:
# 🎯 Exercise 9 — your code here ────────────────────────────────────────────
# 1. Define a small test case.
# 2. Run the suite and inspect the result.

In [ ]:
# ✅ Exercise 9 — Solution ────────────────────────────────────────────────────
class InlineValidationTests(unittest.TestCase):
    def test_invalid_coordinates_raise(self):
        with self.assertRaises(GeoJSONValidationError):
            validate_feature({
                'type': 'Feature',
                'properties': {'city': 'Nowhere'},
                'geometry': {'type': 'Point', 'coordinates': [181.0, 45.0]},
            })

inline_suite = unittest.defaultTestLoader.loadTestsFromTestCase(InlineValidationTests)
unittest.TextTestRunner(verbosity=2).run(inline_suite)

---
## 📖 Section 10 — Batch Processing

Real projects rarely stop at one file. Batch processing applies the same validation rules across a whole folder.

### 10.1 Summarising many input files
The core pattern is: discover files → validate each file → accumulate summary rows → write outputs.

In [ ]:
# 💻 10.1  Process every GeoJSON file in the input folder
# ─────────────────────────────────────────────────────────────────────────────
def batch_validate_geojson_files(input_dir: Path):
    summary_rows = []
    combined_valid_features = []

    for path in sorted(input_dir.glob('*.geojson')):
        try:
            payload = read_json_file(path)
            valid, invalid = split_valid_and_invalid_features(payload)
            summary_rows.append({
                'file_name': path.name,
                'valid_features': len(valid),
                'invalid_features': len(invalid),
            })
            combined_valid_features.extend(valid)
        except GeoFilePipelineError as exc:
            summary_rows.append({
                'file_name': path.name,
                'valid_features': 0,
                'invalid_features': 0,
            })
            logger.error('Batch read failure for %s: %s', path.name, exc)

    return summary_rows, combined_valid_features

batch_rows, combined_valid_features = batch_validate_geojson_files(INPUT_DIR)
pprint(batch_rows)

In [ ]:
# 💻 10.2  Write the batch summary CSV and combined GeoJSON
# ─────────────────────────────────────────────────────────────────────────────
batch_summary_path = OUTPUT_DIR / 'batch_summary.csv'
with batch_summary_path.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['file_name', 'valid_features', 'invalid_features'])
    writer.writeheader()
    writer.writerows(batch_rows)

combined_geojson_path = OUTPUT_DIR / 'combined_valid_points.geojson'
geo_file_tools.save_geojson(combined_valid_features, combined_geojson_path)

print('Batch summary CSV   :', batch_summary_path)
print('Combined valid file :', combined_geojson_path)

### 🎯 Exercise 10 — Report files that need attention

**Task:** Loop through the batch summary rows and print only the files where `invalid_features` is greater than zero.

**Steps:**
1. Iterate over `batch_rows`.
2. Check the `invalid_features` value and print matching rows.

```python
# Hint
if row["invalid_features"] > 0:
    print(row)
```

In [ ]:
# 🎯 Exercise 10 — your code here ────────────────────────────────────────────
# 1. Loop over the summary rows.
# 2. Print rows that contain invalid features.

In [ ]:
# ✅ Exercise 10 — Solution ────────────────────────────────────────────────────
for row in batch_rows:
    if row['invalid_features'] > 0:
        print('Needs attention:', row)

---
## 🔬 Mini-Lab — GeoJSON QA Pipeline

**Scenario:** You have inherited a folder of GeoJSON point files from different colleagues. Some are clean, some contain impossible coordinates, and some mix good and bad data in the same file. Your job is to build a repeatable quality-assurance pipeline.

**Tasks:**
1. Create 3 sample input files (1 clean, 1 with bad coords, 1 mixed)
2. Run batch validation with logging
3. Count valid/invalid per file
4. Save `batch_summary.csv`
5. Build combined valid GeoJSON from all clean features
6. Write a quick unit test for the pipeline

In [ ]:
# 🔬 Step 1 — Create three lab input files
# ─────────────────────────────────────────────────────────────────────────────
lab_files = {
    'lab_clean.geojson': {
        'type': 'FeatureCollection',
        'features': [
            {'type': 'Feature', 'properties': {'city': 'Haarlem'}, 'geometry': {'type': 'Point', 'coordinates': [4.6462, 52.3874]}},
            {'type': 'Feature', 'properties': {'city': 'Leiden'}, 'geometry': {'type': 'Point', 'coordinates': [4.4970, 52.1601]}},
        ],
    },
    'lab_bad_coords.geojson': {
        'type': 'FeatureCollection',
        'features': [
            {'type': 'Feature', 'properties': {'city': 'BadCoord'}, 'geometry': {'type': 'Point', 'coordinates': [250.0, 95.0]}},
        ],
    },
    'lab_mixed.geojson': {
        'type': 'FeatureCollection',
        'features': [
            {'type': 'Feature', 'properties': {'city': 'Zwolle'}, 'geometry': {'type': 'Point', 'coordinates': [6.0944, 52.5168]}},
            {'type': 'Feature', 'properties': {'city': 'BrokenAgain'}, 'geometry': {'type': 'Point', 'coordinates': [-190.0, 12.0]}},
        ],
    },
}

for filename, payload in lab_files.items():
    path = INPUT_DIR / filename
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print('Created', path.name)

In [ ]:
# 🔬 Step 2 — Run batch validation with logging
# ─────────────────────────────────────────────────────────────────────────────
lab_logger = logging.getLogger('gpm.week3.lab')
lab_logger.setLevel(logging.DEBUG)
lab_logger.handlers.clear()
lab_file_handler = logging.FileHandler(log_path, encoding='utf-8')
lab_file_handler.setFormatter(formatter)
lab_logger.addHandler(lab_file_handler)
lab_logger.info('Mini-Lab batch validation started.')

In [ ]:
# 🔬 Step 3 — Count valid and invalid features per file
# ─────────────────────────────────────────────────────────────────────────────
lab_results = []
lab_combined_valid = []
for path in sorted(INPUT_DIR.glob('lab_*.geojson')):
    payload = read_json_file(path)
    valid, invalid = split_valid_and_invalid_features(payload)
    lab_results.append({
        'file_name': path.name,
        'valid_features': len(valid),
        'invalid_features': len(invalid),
    })
    lab_combined_valid.extend(valid)
    lab_logger.info('Mini-Lab processed %s -> valid=%s invalid=%s', path.name, len(valid), len(invalid))

pprint(lab_results)

In [ ]:
# 🔬 Step 4 — Save batch_summary.csv
# ─────────────────────────────────────────────────────────────────────────────
lab_summary_path = OUTPUT_DIR / 'lab_batch_summary.csv'
with lab_summary_path.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['file_name', 'valid_features', 'invalid_features'])
    writer.writeheader()
    writer.writerows(lab_results)

print('Mini-Lab summary saved to:', lab_summary_path)

In [ ]:
# 🔬 Step 5 — Build a combined valid GeoJSON file
# ─────────────────────────────────────────────────────────────────────────────
lab_combined_path = OUTPUT_DIR / 'lab_combined_valid.geojson'
geo_file_tools.save_geojson(lab_combined_valid, lab_combined_path)

print('Combined valid feature count:', len(lab_combined_valid))
print('Combined file path          :', lab_combined_path)

In [ ]:
# 🔬 Step 6 — Write a quick unit test for the pipeline
# ─────────────────────────────────────────────────────────────────────────────
class BatchPipelineTests(unittest.TestCase):
    def test_lab_results_include_invalid_records(self):
        indexed = {row['file_name']: row for row in lab_results}
        self.assertGreater(indexed['lab_bad_coords.geojson']['invalid_features'], 0)
        self.assertEqual(indexed['lab_clean.geojson']['invalid_features'], 0)

unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(BatchPipelineTests)
)

### 🚀 Extension Ideas

- Add polygon and line support to `geo_file_tools.py`
- Include a UUID per output feature for traceability
- Turn the batch pipeline into a command-line script with `argparse`

---
## ✅ Week 3 Summary

| Topic | Key concepts mastered |
|-------|-----------------------|
| pathlib | `Path`, `mkdir`, `glob`, `read_text`, `write_text` |
| csv module | `DictReader`, `DictWriter`, explicit field names |
| json module | `loads`, `dumps`, pretty-printing nested objects |
| GeoJSON structure | `FeatureCollection`, `Feature`, `properties`, `geometry` |
| validation | Coordinate range checks and geometry-type rules |
| modules | Writing and importing `geo_file_tools.py` |
| exceptions | Custom error classes for missing files and malformed JSON |
| logging | `FileHandler`, levels, formatted audit messages |
| unittest | `TestCase`, `setUp`, `assertEqual`, `assertRaises` |
| batch processing | Folder loops, summary CSV output, combined GeoJSON export |

---

### ☑️ Self-assessment checklist

- [ ] Create/read/write files with `pathlib` without hard-coded paths
- [ ] Read a CSV with `csv.DictReader` and convert lon/lat strings to floats
- [ ] Build and save a valid GeoJSON `FeatureCollection` from Python dicts
- [ ] Write `validate_feature()` that raises a custom exception on bad input
- [ ] Add a `FileHandler` logger that writes to `logs/week3.log`
- [ ] Import a function from a local `.py` module you wrote yourself
- [ ] Write a `unittest.TestCase` with at least three test methods
- [ ] Run a batch loop over multiple files and write a CSV summary

---

## 📚 Week 4 Preview

| Topic | What you will learn |
|-------|---------------------|
| coordinate systems and EPSG codes | Why CRS identifiers matter and how to read them |
| WGS84 vs projected CRS | Why degrees and metres answer different questions |
| PyProj Transformer | Safe coordinate transforms with `always_xy=True` |
| Shapely geometry objects | Points, lines, polygons, buffers, and properties |
| topology predicates | `contains`, `within`, `intersects`, and `touches` |

---

## 📖 Further Reading

| Resource | Why |
|----------|-----|
| [Python pathlib docs](https://docs.python.org/3/library/pathlib.html) | Portable path handling and file utilities |
| [Python csv docs](https://docs.python.org/3/library/csv.html) | Official reference for CSV readers and writers |
| [Python json docs](https://docs.python.org/3/library/json.html) | Parse and serialise structured data |
| [GeoJSON RFC 7946](https://www.rfc-editor.org/rfc/rfc7946) | Formal GeoJSON rules and coordinate expectations |
| [Python logging HOWTO](https://docs.python.org/3/howto/logging.html) | Configure handlers, formatters, and log levels |
| [Real Python unittest guide](https://realpython.com/python-testing/) | Friendly introduction to Python testing patterns |

---

*Geospatial Python Mastery — Week 3 of 10*
*For educational use. Please keep feedback to help improve future iterations.*